# 2 · Hasard et statistiques de base — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch02_stats/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Exécuter tout » va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « pourquoi ? » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch02_stats/03_notebook.ipynb`, créée par `python tools/start_chapter.py 2`) : ce fichier-ci est mis à jour par Claude.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 2.1 à 2.7 | vérifier tes exercices papier ✏️ | ✏️ | ★ à ★★ | |
| A | 2.13 à 2.18 | résumer des données : tendances centrales, dispersion, histogramme, lois, 68-95-99,7 (*prochaine session de génération*) | | | |
| B | 2.19 à 2.21 | tirer au hasard : roue de la fortune, dépendance, avec ou sans remise (*prochaine session de génération*) | | | |
| C | 2.22 à 2.24 | le bootstrap et son intervalle de confiance (*prochaine session de génération*) | | | |
| D | 2.25 à 2.32 | grande dimension, covariance et corrélation, ddof, Anscombe (*prochaine session de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="2")  # your own library

## Objectifs et rappel express

- Résumer des données par un centre (moyenne, médiane, mode) et une dispersion (variance, écart-type, percentiles), et savoir lequel choisir.
- Tirer au hasard de façon reproductible, dans une loi discrète, avec ou sans remise.
- Mesurer l'incertitude d'une statistique par le bootstrap.
- Calculer et interpréter covariance et corrélation, et toujours regarder les données.

**Rappel express.** $\bar{x} = \frac{1}{n}\sum_i x_i$ ; $\mathrm{Var}(x) = \frac{1}{n - \mathrm{ddof}}\sum_i (x_i - \bar{x})^2$ et $\sigma = \sqrt{\mathrm{Var}(x)}$ ; $z_i = \frac{x_i - \bar{x}}{\sigma}$ ; loi normale : 68 %, 95 %, 99,7 % des tirages à moins de 1, 2, 3 écarts-types de la moyenne ; $\mathbb{E}[X] = \sum_k x_k\,p_k$ ; $\mathrm{Cov}(x, y) = \frac{1}{n - \mathrm{ddof}}\sum_i (x_i - \bar{x})(y_i - \bar{y})$ et $r = \frac{\mathrm{Cov}(x, y)}{\sigma_x\,\sigma_y}$. En NumPy : `rng = np.random.default_rng(seed)`, `x.var(ddof=...)`, `np.percentile`, `np.cov(x, y)`, `np.corrcoef(x, y)`.

## Partie 0 · Vérifier tes exercices papier (✏️ de 2.1 à 2.7)

Fais d'abord les exercices ✏️ de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée (par exemple `42`, `0.125`, `[7, -2]` ou `"texte"`), pas l'expression Python, sinon tu ne vérifies rien. Arrondis comme l'énoncé le demande. Les réponses pas encore remplies affichent ⏳. L'exercice ∂ 2.8 se corrige avec `05_solutions.md`.

**Ex 2.1 — Moyenne, médiane et mode d'une liste de salaires**

In [ ]:
# Ex 2.1: the values you found on paper
answer_2_1a = ...  # a) the mean salary, rounded to the euro
answer_2_1b = ...  # b) the median
answer_2_1c = ...  # c) the mode
answer_2_1d = ...  # d) how many employees earn LESS than the mean
answer_2_1e = ...  # e) the new mean with 30 000 instead of 12 000, rounded to the euro
answer_2_1f = ...  # f) the new median

for letter, answer in zip("abcdef", [answer_2_1a, answer_2_1b, answer_2_1c, answer_2_1d, answer_2_1e, answer_2_1f]):
    wb.check(f"2.1{letter}", answer)

**Ex 2.2 — De la casse de voitures à la distribution de probabilité**

In [ ]:
# Ex 2.2: the values you found on paper
answer_2_2a = ...  # a) P(minivan), 2 decimals
answer_2_2b = ...  # b) the 5 probabilities in the order of the statement, a list
answer_2_2c = ...  # c) the angle of the minivan slice, in degrees (1 decimal)
answer_2_2d = ...  # d) P(not an SUV), 2 decimals
answer_2_2e = ...  # e) the mean number of pick-ups in 50 draws
answer_2_2f = ...  # f) the cumulative sums of the probabilities, a list
answer_2_2g = ...  # g) the car type chosen by u = 0.71 (a string)

for letter, answer in zip("abcdefg", [answer_2_2a, answer_2_2b, answer_2_2c, answer_2_2d, answer_2_2e, answer_2_2f, answer_2_2g]):
    wb.check(f"2.2{letter}", answer)

**Ex 2.3 — La règle 68-95-99,7 sur les nageoires des manchots**

In [ ]:
# Ex 2.3: the values you found on paper
answer_2_3a = ...  # a) the interval holding about 68 % of the Adelie, a list [start, end] in mm
answer_2_3b = ...  # b) the interval holding about 95 % of them
answer_2_3c = ...  # c) the proportion above 203 mm (3 decimals)
answer_2_3d = ...  # d) the expected number of Adelie above 203 mm among 151 (integer)
answer_2_3e = ...  # e) the z-score of a 210 mm flipper (2 decimals)

for letter, answer in zip("abcde", [answer_2_3a, answer_2_3b, answer_2_3c, answer_2_3d, answer_2_3e]):
    wb.check(f"2.3{letter}", answer)

**Ex 2.4 — Variance : diviser par N ou par N − 1 ?**

In [ ]:
# Ex 2.4: the values you found on paper
answer_2_4a = ...  # a) the mean
answer_2_4b = ...  # b) the sum of the squared deviations
answer_2_4c = ...  # c) the variance with ddof = 0
answer_2_4d = ...  # d) the variance with ddof = 1
answer_2_4e = ...  # e) the standard deviation with ddof = 0 (3 decimals)
answer_2_4f = ...  # f) the standard deviation with ddof = 1 (3 decimals)
answer_2_4g = ...  # g) the mean of the squares
answer_2_4h = ...  # h) the variance (ddof = 0) after adding 100 to every value
answer_2_4i = ...  # i) the standard deviation (ddof = 0) in milliseconds (1 decimal)

for letter, answer in zip("abcdefghi", [answer_2_4a, answer_2_4b, answer_2_4c, answer_2_4d, answer_2_4e, answer_2_4f, answer_2_4g, answer_2_4h, answer_2_4i]):
    wb.check(f"2.4{letter}", answer)

**Ex 2.5 — Espérances : Bernoulli, multinoulli et jeu de hasard**

In [ ]:
# Ex 2.5: the values you found on paper
answer_2_5a = ...  # a) E[X] for the coin (heads = 1)
answer_2_5b = ...  # b) Var(X), 2 decimals
answer_2_5c = ...  # c) the mean number of heads in 200 throws
answer_2_5d = ...  # d) the expected value of a fair 20-sided die
answer_2_5e = ...  # e) the expected class number (1 decimal)
answer_2_5f = ...  # f) the one-hot vector of class 3, a list
answer_2_5g = ...  # g) the limit of the average one-hot vector, a list
answer_2_5h = ...  # h) the mean gain per game, stake deducted (2 decimals)
answer_2_5i = ...  # i) the organiser's mean gain over 400 games, in euros

for letter, answer in zip("abcdefghi", [answer_2_5a, answer_2_5b, answer_2_5c, answer_2_5d, answer_2_5e, answer_2_5f, answer_2_5g, answer_2_5h, answer_2_5i]):
    wb.check(f"2.5{letter}", answer)

**Ex 2.6 — Compter les tirages avec et sans remise**

In [ ]:
# Ex 2.6: the values you found on paper
answer_2_6a = ...  # a) without replacement, order ignored: number of new datasets
answer_2_6b = ...  # b) with replacement, order ignored
answer_2_6c = ...  # c) with replacement, order counted
answer_2_6d = ...  # d) 7 examples, draw 3 without replacement, order ignored
answer_2_6e = ...  # e) number of possible bootstrap sequences of the 5 examples
answer_2_6f = ...  # f) P(a given example is in none of the 5 draws), 4 decimals
answer_2_6g = ...  # g) the same probability for 1 000 draws among 1 000 (4 decimals)
answer_2_6h = ...  # h) the mean share of examples drawn at least once (4 decimals)

for letter, answer in zip("abcdefgh", [answer_2_6a, answer_2_6b, answer_2_6c, answer_2_6d, answer_2_6e, answer_2_6f, answer_2_6g, answer_2_6h]):
    wb.check(f"2.6{letter}", answer)

**Ex 2.7 — Covariance et corrélation de cinq points à la main**

In [ ]:
# Ex 2.7: the values you found on paper
answer_2_7a = ...  # a) the means of x and y, a list
answer_2_7b = ...  # b) the sum of the products of the deviations
answer_2_7c = ...  # c) the covariance (ddof = 0)
answer_2_7d = ...  # d) the covariance with ddof = 1 (2 decimals)
answer_2_7e = ...  # e) the standard deviations of x and y, a list (3 decimals)
answer_2_7f = ...  # f) the correlation r (3 decimals)
answer_2_7g = ...  # g) the covariance with x in minutes
answer_2_7h = ...  # h) the correlation with x in minutes (3 decimals)

for letter, answer in zip("abcdefgh", [answer_2_7a, answer_2_7b, answer_2_7c, answer_2_7d, answer_2_7e, answer_2_7f, answer_2_7g, answer_2_7h]):
    wb.check(f"2.7{letter}", answer)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu choisir entre moyenne et médiane, et calculer variance et écart-type avec le bon diviseur ?
2. Sais-tu utiliser la règle 68-95-99,7 et dire quand elle ne s'applique pas ?
3. Sais-tu calculer une covariance et une corrélation, et dire ce qu'une corrélation ne prouve pas ?

**Pour aller plus loin** : *Seeing Theory* (Brown University) et le cours *Statistiques et probabilités* de Khan Academy, cités dans la fiche. La suite : le ch. 3 (probabilités conditionnelles, mesures de la qualité d'un classifieur), puis le bootstrap revient au ch. 14 (forêts aléatoires) et la matrice de covariance au ch. 12 (analyse en composantes principales).

*Les exercices de code 2.13 à 2.32 (parties A à D, ta librairie `mylearn.stats`) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 2` pour obtenir le notebook complet.*